# Lab 3.4 &mdash; State Reducers

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Declare <i>how</i> a field combines, once, instead of hand-merging in every node
- Decide which fields need a reducer and which should simply be replaced
- Run two nodes in the same step and see what happens without one

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

Look at what every node has been doing:

```python
return {"notes": state["notes"] + ["check"]}
```

Read, append, write the whole thing back. It works, and it puts the merge rule in every node.
A **reducer** moves it into the schema, once:

```python
notes: Annotated[list, add]     # nodes return only their own line; LangGraph appends
```

The default, if you say nothing, is **replace**: last write wins. Right for `decision`, a bug for
`notes` &mdash; and not a matter of taste at all once two nodes run in the same step.

## Section 1 &mdash; Which fields need one

A reducer on every field is as wrong as a reducer on none.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


def merge_approvers(old: list, new: list) -> list:
    """Two nodes may independently name the same approver."""
    keep_all  = old + new
    keep_once = old + [a for a in new if a not in old]
    return keep_once    # each approver once, in the order first required


def needs_a_reducer() -> set:
    """Which of these combine across nodes, rather than being replaced?"""
    # "notes"     -- every node adds its own line
    # "approvers" -- different nodes may each require a sign-off
    # "decision"  -- one sentence; whoever writes last is the answer
    # "days"      -- written once and never again
    return {"notes", "approvers"}

In [ ]:
# --- Self-check: Section 1
check("merge_approvers does not record the same person twice",
      lambda: merge_approvers(["Devi R."], ["HR", "Devi R."]) == ["Devi R.", "HR"])
check("notes and approvers accumulate; decision and days do not",
      lambda: needs_a_reducer() == {"notes", "approvers"},
      "appending decisions would leave you holding every answer the graph considered")
score()

## Section 2 &mdash; Two nodes at the same time

Both `check_balance` and `check_calendar` have an edge from `summarise`, so LangGraph runs them
**in one step**. Both write `notes` and `approvers`.

Notice how much simpler the nodes got: they return their own line, and nothing reads
`state["notes"]` any more.

In [ ]:
from operator import add

class RunState(TypedDict):
    request_id: str
    days: int
    notes: Annotated[list, add]                  # declared once, obeyed by every node
    approvers: Annotated[list, merge_approvers]
    decision: str                                # no reducer: replaced


def summarise(state):
    return {"days": REQUESTS[state["request_id"]]["days"], "notes": ["summarise"]}

def check_balance(state):
    out = {"notes": ["check_balance"]}
    if state["days"] > POLICY["manager_over_days"]:
        out["approvers"] = [REQUESTS[state["request_id"]]["manager"]]
    return out

def check_calendar(state):
    out = {"notes": ["check_calendar"]}
    if state["days"] > POLICY["manager_over_days"]:          # concludes the same thing
        out["approvers"] = [REQUESTS[state["request_id"]]["manager"]]
    if state["days"] > POLICY["hr_over_days"]:
        out["approvers"] = out.get("approvers", []) + ["HR"]
    return out

def finalise(state):
    return {"decision": f'sign-off from: {", ".join(state["approvers"]) or "nobody"}',
            "notes": ["finalise"]}


def build_parallel_graph():
    builder = StateGraph(RunState)
    for name, fn in [("summarise", summarise), ("check_balance", check_balance),
                     ("check_calendar", check_calendar), ("finalise", finalise)]:
        builder.add_node(name, fn)
    builder.add_edge(START, "summarise")
    builder.add_edge("summarise", "check_balance")     # two edges out of one node,
    builder.add_edge("summarise", "check_calendar")    # so both run in the same step
    builder.add_edge("check_balance", "finalise")
    builder.add_edge("check_calendar", "finalise")
    builder.add_edge("finalise", END)
    return builder.compile()

In [ ]:
# --- Self-check: Section 2   (a real fan-out, really executed)
def run(rid):
    return build_parallel_graph().invoke({"request_id": rid, "notes": [], "approvers": []})

check("both parallel nodes' notes survive -- four nodes, four notes",
      lambda: len(run("LV-5004")["notes"]) == 4
          and {"check_balance", "check_calendar"} <= set(run("LV-5004")["notes"]),
      "without a reducer on notes, one would have overwritten the other")
check("LV-5004 needs the manager once, not twice, plus HR",
      lambda: run("LV-5004")["approvers"] == ["Sam O.", "HR"],
      "both nodes named the manager; merge_approvers is what stops the duplicate")
score()

## Watch it run &mdash; and watch it fail without one

The same fan-out, with `notes` declared as a plain `list`.

In [ ]:
def with_reducers():
    out = build_parallel_graph().invoke({"request_id": "LV-5004", "notes": [], "approvers": []})
    print("with reducers:", out["notes"], "|", out["approvers"], "|", out["decision"])

guard(with_reducers)          # a reducer raises at INVOKE time, not at build time

class NoReducer(TypedDict):
    request_id: str
    days: int
    notes: list            # plain list: nothing says how two writes combine
    approvers: list
    decision: str

b = StateGraph(NoReducer)
b.add_node("summarise", summarise)
b.add_node("a", lambda s: {"notes": ["a"]})
b.add_node("b", lambda s: {"notes": ["b"]})
b.add_edge(START, "summarise")
b.add_edge("summarise", "a"); b.add_edge("summarise", "b")
b.add_edge("a", END); b.add_edge("b", END)

try:
    print("without   :", b.compile().invoke({"request_id": "LV-5004", "notes": [], "approvers": []})["notes"])
except Exception as exc:
    print(f"without   : {type(exc).__name__}: {str(exc)[:150]}")

### Read it

LangGraph **refuses** rather than picking a winner: two nodes wrote the same key in one step and
nothing said how to combine them.

Remember that, because the sequential version of this bug is not loud. In a chain, a node
returning `{"notes": ["mine"]}` on a field with no reducer silently throws away everything before
it, and you find out when the audit trail has one line in it.

**The habit:** decide the merge rule where you declare the field, not in each node that writes it.

In [ ]:
score()

## Your turn

1. Replace `merge_approvers` with `add` and re-run. Most LangGraph code you read uses
   `Annotated[list, add]` &mdash; now you know what it means and when it is not enough.
2. Add a third parallel node that also writes `days`. Predict what happens, then run it.